# Titanic Survival Prediction — Final Predictions

## Objective

Train the selected Logistic Regression pipeline on the full
labeled Titanic dataset and generate predictions for Kaggle's
unlabeled test dataset.

The predictions will be saved in submission.csv.

In [1]:
import pandas as pd

train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)

Training dataset: (891, 12)
Test dataset: (418, 11)


In [2]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X_train_full = train_df[features]
y_train_full = train_df["Survived"]

X_test = test_df[features]

print("Training features:", X_train_full.shape)
print("Training target:", y_train_full.shape)
print("Test features:", X_test.shape)

Training features: (891, 7)
Training target: (891,)
Test features: (418, 7)


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numerical_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=1000
    ))
])

print("Final pipeline created successfully!")

Final pipeline created successfully!


In [4]:
final_model.fit(X_train_full, y_train_full)

print("Final model trained successfully!")

Final model trained successfully!


In [5]:
test_predictions = final_model.predict(X_test)

print("Number of predictions:", len(test_predictions))
print("First 10 predictions:", test_predictions[:10])

Number of predictions: 418
First 10 predictions: [0 0 0 0 1 0 1 0 1 0]


In [6]:
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions
})

print(submission.head())
print("Submission shape:", submission.shape)

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Submission shape: (418, 2)


In [7]:
submission.to_csv(
    "../outputs/submission.csv",
    index=False
)

print("Submission file saved successfully!")

Submission file saved successfully!


In [8]:
print("Missing values:")
print(submission.isnull().sum())

print("\nPrediction counts:")
print(submission["Survived"].value_counts())

print("\nUnique prediction values:")
print(submission["Survived"].unique())

print("\nSubmission shape:")
print(submission.shape)

print("\nPassenger IDs unique:")
print(submission["PassengerId"].is_unique)

Missing values:
PassengerId    0
Survived       0
dtype: int64

Prediction counts:
Survived
0    262
1    156
Name: count, dtype: int64

Unique prediction values:
[0 1]

Submission shape:
(418, 2)

Passenger IDs unique:
True


## Kaggle Competition Results

The final Logistic Regression model was trained on all
891 labeled passengers and used to predict survival
for the 418 passengers in the test dataset.

### Results

- Local validation accuracy: 80.45%
- Kaggle public leaderboard accuracy: 76.555%
- Model: Logistic Regression
- Regularization parameter: C = 1.0

### Key Takeaways

The model achieved a Kaggle public leaderboard score
of 76.555%, demonstrating its ability to make
predictions on previously unseen data.

The difference between local validation and Kaggle
performance highlights the importance of evaluating
machine learning models on independent datasets.